# 01 - Opening I3 Files And Looking Inside

This first notebook is intentionally explicit. Instead of hiding paths and helper functions in another file, we write the important pieces here so you can see what Python is doing.

In this notebook you will learn to:

1. Point Python at an IceCube `.i3.zst` file.
2. Open that file with `dataio.I3File`.
3. Count the different kinds of frames in the file.
4. Print the keys stored inside a Physics frame.


## The files we will use

IceCube event data is usually split into an event file plus a matching GCD file. GCD means Geometry, Calibration, and DetectorStatus. The event file contains event-by-event information; the GCD file describes the detector for that run or simulation set.


In [ ]:
from pathlib import Path
from collections import Counter

from icecube import dataio

SIM_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133575/Level2_IC86.2019_data_Run00133575_0101_78_503_GCD.i3.zst')
SIM_FILE = Path('/data/sim/IceCube/2020/filtered/level2/CORSIKA-in-ice/20904/0000000-0000999/Level2_IC86.2020_corsika.020904.000000.i3.zst')

DATA_GCD = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_0101_78_503_GCD.i3.zst')
DATA_FILE = Path('/data/exp/IceCube/2020/filtered/level2/0101/Run00133576/Level2_IC86.2019_data_Run00133576_Subrun00000000_00000000.i3.zst')

for label, path in [('simulation GCD', SIM_GCD), ('simulation events', SIM_FILE), ('data GCD', DATA_GCD), ('data events', DATA_FILE)]:
    print(f'{label:18s}: exists={path.exists()}')
    print(f'  {path}')


## A tiny frame-stop helper

An I3 file is a stream of frames. Each frame has a stop, which tells you what kind of frame it is. In this IceTray environment, Physics frames show up as `P`, DAQ frames as `Q`, Geometry as `G`, Calibration as `C`, and DetectorStatus as `D`.

The function below converts those short labels to names that are easier to read. It is small enough that you should be able to understand and modify it.


In [ ]:
def frame_stop(frame):
    """Return a readable name for the frame stop."""
    stop = str(frame.Stop)
    names = {
        'I': 'TrayInfo',
        'G': 'Geometry',
        'C': 'Calibration',
        'D': 'DetectorStatus',
        'Q': 'DAQ',
        'P': 'Physics',
    }
    return names.get(stop, stop)

print('If IceTray gives us P, we will call it:', frame_stop(type('FakeFrame', (), {'Stop': 'P'})()))


## Count frame types

This cell opens the experimental event file and reads the first 100 frames. It counts how many frames of each type appear. Notice that we close the file when we are finished.


In [ ]:
counts = Counter()
i3_file = dataio.I3File(str(DATA_FILE))

for frame_number in range(100):
    if not i3_file.more():
        break
    frame = i3_file.pop_frame()
    counts[frame_stop(frame)] += 1

i3_file.close()

print('Frame types in the first 100 frames:')
for stop, count in counts.items():
    print(f'  {stop:14s} {count}')


## Inspect one Physics frame

A frame is like a dictionary: it contains named objects. The names are called keys. The code below finds the first Physics frame and prints the first several keys.

This is one of the most useful early IceTray habits: before assuming a key exists, print the keys and look.


In [ ]:
physics_frame = None
i3_file = dataio.I3File(str(DATA_FILE))

while i3_file.more():
    frame = i3_file.pop_frame()
    if frame_stop(frame) == 'Physics':
        physics_frame = frame
        break

i3_file.close()

if physics_frame is None:
    raise RuntimeError('No Physics frame was found. Try checking the file path or reading more frames.')

print('Found a Physics frame.')
print(f'This frame has {len(physics_frame.keys())} keys. Here are the first 40:')
for key in list(physics_frame.keys())[:40]:
    print(' ', key, '->', type(physics_frame[key]).__name__)


## Pull out the event header

Many Physics frames contain `I3EventHeader`. This object stores bookkeeping information such as run number and event number.


In [ ]:
if 'I3EventHeader' in physics_frame:
    header = physics_frame['I3EventHeader']
    print('Run ID:        ', header.run_id)
    print('Event ID:      ', header.event_id)
    print('Sub-event ID:  ', header.sub_event_id)
    print('Sub-event type:', header.sub_event_stream)
else:
    print('This frame does not contain I3EventHeader.')


## Look for useful keys

The next cell searches the key names for words we expect to matter in later notebooks: pulses, filters, reconstructions, and simulation truth.


In [ ]:
interesting_words = ['pulse', 'filter', 'spline', 'mpe', 'linefit', 'energy', 'mc', 'primary']

print('Interesting-looking keys:')
for key in physics_frame.keys():
    if any(word in key.lower() for word in interesting_words):
        print(' ', key)


## Practice

1. Change `DATA_FILE` to `SIM_FILE` and rerun the frame-counting cell.
2. Open the GCD file instead of the event file. Which frame types do you see?
3. In the first Physics frame, find one pulse key and one reconstruction key.
4. Use `dataio-shovel` in a terminal and compare what you see there to what Python prints here.
